In [1]:
# cargar librerías básicas
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, shapiro
import pandas as pd

In [2]:
df = pd.read_csv("limpio_porfin.csv")

In [4]:
df_reduced = df.drop(
    ["codigoGrasa", "subtitulo", "beneficios", "aplicaciones",
     "categoria", "idDatosGrasas", "Corrosión al Cobre",
     "Indice de Carga-Desgaste", "Registro NSF"],
    axis=1
)


In [5]:
df_reduced.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 17 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Aceite Base                               47 non-null     object 
 1   Espesante                                 45 non-null     object 
 2   Grado NLGI Consistencia                   51 non-null     float64
 3   Viscosidad del Aceite Base a 40°C. cSt    50 non-null     float64
 4   Penetración de Cono a 25°C, 0.1mm         51 non-null     float64
 5   Punto de Gota, °C                         51 non-null     float64
 6   Estabilidad Mecánica, %                   51 non-null     float64
 7   Punto de Soldadura Cuatro Bolas, kgf      51 non-null     float64
 8   Desgaste Cuatro Bolas, mm                 51 non-null     float64
 9   Carga Timken Ok, lb                       51 non-null     int64  
 10  Resistencia al Lavado por Agua a 80°C, %

Penetración de Cono a 25°C, 0.1mm - average
Punto de Gota, °C - valor + std
Estabilidad Mecánica, % - +,- stdev para rango. media para rellenar
Punto de Soldadura Cuatro Bolas, kgf - average 
Desgaste Cuatro Bolas, mm - mediana porque había muchos vacíos
Carga Timken Ok, lb  - mediana porque etaban en múltiplos de 5
Resistencia al Lavado por Agua a 80°C - media para rellenar, desv para rangos
factor de velocidad - mediana 


In [6]:
# 1. Separar columnas numéricas y categóricas
numeric_cols = df_reduced.select_dtypes(include=['number']).columns
categorical_cols = df_reduced[
    ["Aceite Base", "Espesante", "descripcion", "color", "textura"]
]

print("Numéricas:", list(numeric_cols))
print("Categóricas:", list(categorical_cols))

Numéricas: ['Grado NLGI Consistencia', 'Viscosidad del Aceite Base a 40°C. cSt', 'Penetración de Cono a 25°C, 0.1mm', 'Punto de Gota, °C', 'Estabilidad Mecánica, %', 'Punto de Soldadura Cuatro Bolas, kgf', 'Desgaste Cuatro Bolas, mm', 'Carga Timken Ok, lb', 'Resistencia al Lavado por Agua a 80°C, %', 'Factor de Velocidad', 'Temperatura de Servicio °C, min', 'Temperatura de Servicio °C, max']
Categóricas: ['Aceite Base', 'Espesante', 'descripcion', 'color', 'textura']


In [8]:
df_reduced.to_csv("final_df_final.csv")

### 1. Recomendador simple

Construir un score técnico compuesto usando:

- Punto de soldadura (más alto = mejor)

- Desgaste (mm) (más bajo = mejor)

- Punto de gota (°C) (más alto = mejor)

- Estabilidad mecánica (%) (más alto = mejor)

Score normalizado que te permita obtener un TOP de mejores grasas.

##### Ponderaciones

**Análogo al recomendador de pelis:**

Punto soldadura (40%) → Factor determinante como “popularidad”

Desgaste (30%) → Similar a “calidad promedio”

Punto de gota (20%) → Fuerte indicador termo-mecánico

Estabilidad mecánica (10%) → Factor complementario

In [ ]:
# Copia del dataframe
df_simple = pd.read_csv("final_df_final.csv").copy()

# Normalizamos variables (0–1)
def normalize(col):
    return (col - col.min()) / (col.max() - col.min())

df_simple["soldadura_norm"] = normalize(df_simple["Punto de Soldadura Cuatro Bolas, kgf"])
df_simple["gota_norm"] = normalize(df_simple["Punto de Gota, °C"])
df_simple["estabilidad_norm"] = normalize(df_simple["Estabilidad Mecánica, %"])

# Desgaste: como menor es mejor, invertimos el sentido
df_simple["desgaste_norm"] = 1 - normalize(df_simple["Desgaste Cuatro Bolas, mm"])

# Creamos el score final ponderado
df_simple["score"] = (
      df_simple["soldadura_norm"] * 0.4 +
      df_simple["desgaste_norm"] * 0.3 +
      df_simple["gota_norm"] * 0.2 +
      df_simple["estabilidad_norm"] * 0.1
)

In [10]:
# obtener el TOP 20 de mejores grasas
top_20 = df_simple.sort_values("score", ascending=False).head(20)
top_20[[
    "Aceite Base",
    "Espesante",
    "Punto de Soldadura Cuatro Bolas, kgf",
    "Desgaste Cuatro Bolas, mm",
    "Punto de Gota, °C",
    "Estabilidad Mecánica, %",
    "score"
]]


,Aceite Base,Espesante,"Punto de Soldadura Cuatro Bolas, kgf","Desgaste Cuatro Bolas, mm","Punto de Gota, °C","Estabilidad Mecánica, %",score
42,NaN,Complejo de Litio,900.0,0.40,304.0,2.1,0.851464
11,Mineral HT,Complejo Sulfonato de Calcio,900.0,0.50,304.0,3.8,0.746455
9,Mineral HT,Complejo Sulfonato de Calcio,900.0,0.50,304.0,2.6,0.735873
1,Mineral HT,Complejo Sulfonato de Calcio,900.0,0.50,304.0,2.6,0.735873
8,Mineral HT,Complejo Sulfonato de Calcio,900.0,0.50,304.0,1.6,0.727055
7,Mineral HT,Complejo Sulfonato de Calcio,900.0,0.50,304.0,1.6,0.727055
10,Mineral HT,Complejo Sulfonato de Calcio,900.0,0.50,304.0,1.5,0.726173
26,Mineral USP,Complejo Sulfonato de Calcio,500.0,0.35,304.0,1.4,0.659137
27,Mineral USP,Complejo Sulfonato de Calcio,500.0,0.35,304.0,1.4,0.659137
39,Mineral HT,Complejo Sulfonato de Calcio,620.0,0.45,304.0,4.4,0.639438


#### Recomendación basado en similitud de coseno

In [19]:
df2 = pd.read_csv("final_df_final.csv")

In [14]:
import pandas as pd
import numpy as np

# 1. Definir el DataFrame y la columna (EJEMPLO)
# Supongamos que tienes un DataFrame llamado 'df'
# df = pd.DataFrame(...)

# Cambia 'nombre_df' por el nombre de tu DataFrame


# 2. Extraer la columna y calcular la Moda
# La función mode() devuelve una Serie. Usamos [0] para obtener el primer valor si hay una moda única.
# Si hay varias modas (multimodal), se usa el primer valor.
espesante = 'Espesante' # ¡Cambia esto!
moda_columna = df2['Espesante'].mode()[0]

# Opcional: Imprimir la moda calculada para verificación
print(f"La moda de la columna '{espesante}' es: {moda_columna}")

# 3. Rellenar los valores faltantes (NaN) con la Moda
# Usamos el método fillna()
df['Espesante'] = df['Espesante'].fillna(moda_columna)

# Opcional: Verificar si quedan valores nulos en esa columna
print(f"Valores nulos después de rellenar: {df['Espesante'].isnull().sum()}")

La moda de la columna 'Espesante' es: Complejo Sulfonato de Calcio
Valores nulos después de rellenar: 0


In [16]:
moda_columna2 = df2['Aceite Base'].mode()[0]
moda_columna2

'Mineral HT'

In [20]:
import pandas as pd
import numpy as np

# Supongamos que tu DataFrame se llama 'df'
# ¡Asegúrate de cambiar 'df' si tu DataFrame tiene otro nombre!

# 1. Calcular el número de valores faltantes (NaN) por columna
missing_values_count = df2.isnull().sum()

# 2. Filtrar para mostrar solo las columnas con valores faltantes
# La condición > 0 asegura que solo se muestren las columnas donde la suma de nulos sea mayor a cero.
missing_columns = missing_values_count[missing_values_count > 0]

# 3. Imprimir el resultado
print("Columnas con valores faltantes y su conteo:")
print("-" * 40)
print(missing_columns)

Columnas con valores faltantes y su conteo:
----------------------------------------
Series([], dtype: int64)
